# Baseline 01: PromptGuard 2 (Encoder Classifiers)

**Proposal reference:** Section 3 (Evaluation Protocol → Baselines, paragraph "Encoder classifier"),  
Section 2.1 (Encoder-Based Classifiers, `\\sec:encoder`).

This notebook scores every example in `data/eval_proposal/eval.jsonl` with two encoder classifiers:

1. **Primary (gated):** `meta-llama/Llama-Prompt-Guard-2-86M` — mDeBERTa-v3 86M, requires HuggingFace license acceptance + `HF_TOKEN` env var.
2. **Fallback (ungated, always runs):** `protectai/deberta-v3-base-prompt-injection-v2` — DeBERTa-v3-base, no auth needed.

Outputs per spec `BASELINE_SPEC.md §predictions/<detector>.jsonl`:
- `results/baselines/predictions/promptguard2_86m.jsonl`
- `results/baselines/predictions/protectai_deberta_v2.jsonl`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── pip install cell ──────────────────────────────────────────────────────────
# Uncomment and run on Colab or fresh environments.
# On the local conda env (open_prompt_injection) these are already installed.
#
# !pip install transformers>=4.42.0 torch>=2.3.0 huggingface_hub tqdm

## Configuration

All tunable knobs live here. `RUN_MODE` controls per-source example caps  
(smoke=50, medium=500, full=proposal targets). `SEED=3131` matches the dataset build seed.

In [3]:
import os
import sys

# ── Run mode ─────────────────────────────────────────────────────────────────
RUN_MODE = "full"   # "smoke" | "medium" | "full"
SEED = 3131
BATCH_SIZE = 16      # Adjust down (e.g. 8) if OOM on CPU/MPS

# Per-source example caps (BASELINE_SPEC.md §config)
CAPS = {
    "smoke":  50,
    "medium": 500,
    "full":   None,  # No cap — use all examples
}
SOURCE_CAP = CAPS[RUN_MODE]

# ── Device auto-detect: cuda > mps > cpu ─────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"RUN_MODE : {RUN_MODE}")
print(f"SEED     : {SEED}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"DEVICE   : {DEVICE}")
print(f"SOURCE_CAP: {SOURCE_CAP}")

RUN_MODE : full
SEED     : 3131
BATCH_SIZE: 16
DEVICE   : cuda
SOURCE_CAP: None


## Paths

In [4]:
import pathlib

# ── Environment detection: Google Colab vs local ─────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount Google Drive (no-op if already mounted)
    from google.colab import drive
    drive.mount('/content/drive')
    CASCADE_ROOT = pathlib.Path('/content/drive/MyDrive/Thesis')
else:
    # Local run: env override or default local repo layout
    CASCADE_ROOT = pathlib.Path(os.environ.get(
        'CASCADE_ROOT',
        '/Users/lenguyenminhhuy/study/thesis/experiments/cascade-pid'
    ))

DATA_DIR    = CASCADE_ROOT / 'data'
TRAIN_DIR   = DATA_DIR / 'train_proposal'   # Colab: /content/drive/MyDrive/Thesis/data/train_proposal
EVAL_DIR    = DATA_DIR / 'eval_proposal'    # Colab: /content/drive/MyDrive/Thesis/data/eval_proposal
EVAL_PATH   = EVAL_DIR / 'eval.jsonl'
PRED_DIR    = CASCADE_ROOT / 'results' / 'baselines' / 'predictions'
METRICS_DIR = CASCADE_ROOT / 'results' / 'baselines' / 'metrics'

PRED_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

PRED_PG2      = PRED_DIR / 'promptguard2_86m.jsonl'
PRED_DEBERTA  = PRED_DIR / 'protectai_deberta_v2.jsonl'

print(f"IN_COLAB   : {IN_COLAB}")
print(f"CASCADE_ROOT: {CASCADE_ROOT}")
print(f"TRAIN_DIR  : {TRAIN_DIR} (exists: {TRAIN_DIR.exists()})")
print(f"EVAL_PATH  : {EVAL_PATH}")
print(f"PRED_DIR   : {PRED_DIR}")
print(f"eval.jsonl exists: {EVAL_PATH.exists()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB   : True
CASCADE_ROOT: /content/drive/MyDrive/Thesis
TRAIN_DIR  : /content/drive/MyDrive/Thesis/data/train_proposal (exists: True)
EVAL_PATH  : /content/drive/MyDrive/Thesis/data/eval_proposal/eval.jsonl
PRED_DIR   : /content/drive/MyDrive/Thesis/results/baselines/predictions
eval.jsonl exists: True


## Load eval.jsonl — fail fast if missing

Per `BASELINE_SPEC.md`: _"model notebooks fail fast with a clear message if [eval.jsonl] doesn't [exist]."_

In [5]:
import json
import random
from collections import Counter

if not EVAL_PATH.exists():
    raise FileNotFoundError(
        f"\n"
        f"  eval.jsonl not found at:\n"
        f"    {EVAL_PATH}\n"
        f"  Run notebook 00_eval_dataset.ipynb first to build the eval set.\n"
        f"  (Dataset teammate may still be building this in parallel.)"
    )

random.seed(SEED)

# Load all records
records = []
with open(EVAL_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# Apply per-source cap in RUN_MODE smoke / medium
if SOURCE_CAP is not None:
    by_source = {}
    for r in records:
        by_source.setdefault(r['source'], []).append(r)
    capped = []
    for src, rows in by_source.items():
        random.shuffle(rows)
        capped.extend(rows[:SOURCE_CAP])
    records = capped

print(f"Loaded {len(records)} records (RUN_MODE={RUN_MODE}, cap={SOURCE_CAP})")
print("Label distribution:", Counter(r['label'] for r in records))
print("Channel distribution:", Counter(r.get('channel') for r in records))
print("Source distribution:", Counter(r['source'] for r in records))

Loaded 25747 records (RUN_MODE=full, cap=None)
Label distribution: Counter({0: 18634, 1: 7113})
Channel distribution: Counter({None: 18634, 'direct': 2830, 'document': 2819, 'tool': 1464})
Source distribution: Counter({'lmsys': 9452, 'dolly': 4891, 'natural_instructions': 4291, 'struq_synthetic': 2830, 'openpromptinjection': 2819, 'agentdojo': 1464})


## Label-mapping decisions

### PromptGuard 2 (`meta-llama/Llama-Prompt-Guard-2-86M`)

PromptGuard 2 is an mDeBERTa-v3-based classifier trained to detect **jailbreak** and **prompt injection** inputs.  
From the HuggingFace model card and Meta's published usage:

- **Label 0 (`BENIGN`):** Safe / benign input.
- **Label 1 (`INJECTION`):** Prompt injection (indirect, from external data).
- **Label 2 (`JAILBREAK`):** Direct jailbreak attempt.

**Decision:** We treat both label 1 and label 2 as "injection" (positive class), since our eval schema  
uses a binary `label` (0=benign, 1=injection) that covers both injection and jailbreak techniques.  
`score = 1 - P(BENIGN)` = probability of any malicious class.  
`pred = 1 if argmax ∈ {INJECTION, JAILBREAK} else 0`.

### ProtectAI DeBERTa v2 (`protectai/deberta-v3-base-prompt-injection-v2`)

Two-class classifier:
- **Label `SAFE`:** Benign input.
- **Label `INJECTION`:** Prompt injection.

**Decision:** `score = P(INJECTION)`, `pred = argmax == 'INJECTION'`.

### Truncation handling for 512-token encoders

Both models are 512-token encoders. Long inputs (e.g. document-channel injections with long context)  
exceed this limit. We use **overlapping sliding windows** with a stride of 256 tokens and take the  
**maximum injection score** across all windows. This is the standard deployment pattern because:

1. An injection embedded anywhere in a document should trigger detection.
2. Taking the max is conservative — it flags if any window looks malicious.
3. Averaging would dilute the signal from a short injected snippet in a long benign document.

Window parameters: `max_length=512`, `stride=256`, overlap=256 tokens.

## Shared inference utilities

In [6]:
import time
import math
from typing import List, Dict, Any, Optional

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification


MAX_LENGTH = 512
STRIDE = 256


def sliding_window_score(
    text: str,
    tokenizer,
    model,
    injection_class_indices: List[int],
    device: str,
    max_length: int = MAX_LENGTH,
    stride: int = STRIDE,
) -> float:
    """
    Tokenize `text` with a sliding window (window=max_length, stride=stride).
    Run each window through `model` and return the MAX injection-class probability
    across all windows.

    This is the standard deployment pattern for 512-token encoders on long inputs:
    an injection embedded anywhere in a document should trigger detection, so we
    take the most conservative (highest) score.

    Args:
        injection_class_indices: label indices that map to "injection" in id2label.
    Returns:
        float in [0, 1]: maximum injection probability across all windows.
    """
    tokens = tokenizer(
        text,
        return_tensors='pt',
        truncation=False,
        add_special_tokens=True,
    )
    input_ids = tokens['input_ids'][0]  # shape: [seq_len]
    seq_len = input_ids.shape[0]

    if seq_len <= max_length:
        # Fast path: fits in one window
        windows = [input_ids]
    else:
        # Sliding windows: [CLS] ... tokens ... [SEP]
        # We preserve CLS (position 0) and SEP (last token) in each window.
        cls_id = input_ids[0].unsqueeze(0)
        sep_id = input_ids[-1].unsqueeze(0)
        body = input_ids[1:-1]  # strip CLS and SEP
        body_max = max_length - 2  # space for CLS + SEP
        windows = []
        start = 0
        while start < len(body):
            chunk = body[start : start + body_max]
            windows.append(torch.cat([cls_id, chunk, sep_id]))
            if start + body_max >= len(body):
                break
            start += (body_max - (max_length - stride - 2))  # stride in body tokens

    max_inj_score = 0.0
    model.eval()
    with torch.no_grad():
        for window_ids in windows:
            window_ids = window_ids.unsqueeze(0).to(device)  # [1, window_len]
            attention_mask = torch.ones_like(window_ids)
            outputs = model(input_ids=window_ids, attention_mask=attention_mask)
            probs = F.softmax(outputs.logits, dim=-1)[0]  # [num_labels]
            inj_prob = sum(probs[i].item() for i in injection_class_indices)
            if inj_prob > max_inj_score:
                max_inj_score = inj_prob

    return max_inj_score


def batch_score(
    texts: List[str],
    tokenizer,
    model,
    injection_class_indices: List[int],
    device: str,
    batch_size: int = BATCH_SIZE,
    max_length: int = MAX_LENGTH,
    stride: int = STRIDE,
) -> List[Dict[str, Any]]:
    """
    Score a list of texts. Returns list of dicts with keys:
      score (float), pred (int), latency_ms (float).

    Latency is measured as batch wall-clock / batch_size (per-input estimate).
    For texts that require sliding windows, they are scored individually (the
    batch path uses truncation=True for the fast per-text case with padding).

    We use a hybrid approach:
    - Short texts (fit in max_length): batched inference with padding.
    - Long texts (exceed max_length): individual sliding-window inference.
    """
    results = [None] * len(texts)

    # Separate short and long texts
    short_indices = []
    long_indices = []
    for i, text in enumerate(texts):
        enc = tokenizer(text, truncation=False, add_special_tokens=True)
        if len(enc['input_ids']) <= max_length:
            short_indices.append(i)
        else:
            long_indices.append(i)

    # --- Batched inference for short texts ---
    for batch_start in range(0, len(short_indices), batch_size):
        batch_idx = short_indices[batch_start : batch_start + batch_size]
        batch_texts = [texts[i] for i in batch_idx]
        t0 = time.perf_counter()
        enc = tokenizer(
            batch_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        model.eval()
        with torch.no_grad():
            outputs = model(**enc)
        probs = F.softmax(outputs.logits, dim=-1)  # [batch, num_labels]
        t1 = time.perf_counter()
        batch_lat_ms = (t1 - t0) * 1000.0
        per_input_lat_ms = batch_lat_ms / len(batch_idx)

        for k, orig_i in enumerate(batch_idx):
            prob_vec = probs[k]  # [num_labels]
            inj_score = sum(prob_vec[j].item() for j in injection_class_indices)
            pred_label_idx = prob_vec.argmax().item()
            pred = 1 if pred_label_idx in injection_class_indices else 0
            results[orig_i] = {
                'score': round(inj_score, 6),
                'pred': pred,
                'latency_ms': round(per_input_lat_ms, 3),
            }

    # --- Sliding-window inference for long texts (individual) ---
    for orig_i in long_indices:
        text = texts[orig_i]
        t0 = time.perf_counter()
        inj_score = sliding_window_score(
            text, tokenizer, model, injection_class_indices, device,
            max_length=max_length, stride=stride,
        )
        t1 = time.perf_counter()
        lat_ms = (t1 - t0) * 1000.0
        # For sliding-window, argmax at the max-score window already factored in
        pred = 1 if inj_score >= 0.5 else 0
        results[orig_i] = {
            'score': round(inj_score, 6),
            'pred': pred,
            'latency_ms': round(lat_ms, 3),
        }

    return results


def write_predictions_atomic(pred_path: pathlib.Path, prediction_rows: List[dict]):
    """Write predictions jsonl atomically via a .tmp file."""
    tmp_path = pred_path.with_suffix('.jsonl.tmp')
    with open(tmp_path, 'w') as f:
        for row in prediction_rows:
            f.write(json.dumps(row) + '\n')
    tmp_path.rename(pred_path)
    print(f"Written {len(prediction_rows)} predictions → {pred_path}")


print("Inference utilities loaded.")

Inference utilities loaded.


## Fallback: ProtectAI DeBERTa v2 (ungated — always runs)

**Proposal reference:** Section 2.1 — _"ProtectAI's deberta-v3-base-prompt-injection-v2 follows the same  
design, reporting 95% F1 on its own evaluation set."_

This model is ungated (public HuggingFace) so it always runs regardless of HF auth status.  
Label mapping: `SAFE → 0`, `INJECTION → 1`.  
`score = P(INJECTION)`, which is the continuous confidence for the positive class.

In [7]:
PROTECTAI_MODEL_ID = 'protectai/deberta-v3-base-prompt-injection-v2'

print(f"Loading {PROTECTAI_MODEL_ID} ...")
t_load_start = time.perf_counter()

pa_tokenizer = AutoTokenizer.from_pretrained(PROTECTAI_MODEL_ID)
pa_model = AutoModelForSequenceClassification.from_pretrained(PROTECTAI_MODEL_ID)
pa_model = pa_model.to(DEVICE)

t_load_end = time.perf_counter()
print(f"Loaded in {t_load_end - t_load_start:.1f}s")
print(f"Labels: {pa_model.config.id2label}")

# Identify injection class index
# Expected: {0: 'SAFE', 1: 'INJECTION'} or similar — verify at runtime
pa_id2label = pa_model.config.id2label
PA_INJECTION_INDICES = [
    idx for idx, lbl in pa_id2label.items()
    if lbl.upper() in ('INJECTION', 'PROMPT_INJECTION', 'MALICIOUS')
]
if not PA_INJECTION_INDICES:
    raise ValueError(
        f"Could not auto-detect injection label index from id2label: {pa_id2label}\n"
        "Update PA_INJECTION_INDICES manually."
    )
print(f"Injection class indices (ProtectAI): {PA_INJECTION_INDICES} "
      f"→ {[pa_id2label[i] for i in PA_INJECTION_INDICES]}")

Loading protectai/deberta-v3-base-prompt-injection-v2 ...


config.json:   0%|          | 0.00/994 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loaded in 7.2s
Labels: {0: 'SAFE', 1: 'INJECTION'}
Injection class indices (ProtectAI): [1] → ['INJECTION']


In [8]:
from tqdm.auto import tqdm

print(f"Scoring {len(records)} examples with {PROTECTAI_MODEL_ID} ...")
print(f"  BATCH_SIZE={BATCH_SIZE}, DEVICE={DEVICE}")

texts = [r['text'] for r in records]

# Score in BATCH_SIZE chunks with progress bar
pa_raw_results = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='protectai-deberta'):
    batch = texts[i : i + BATCH_SIZE]
    batch_results = batch_score(
        batch,
        pa_tokenizer,
        pa_model,
        PA_INJECTION_INDICES,
        DEVICE,
        batch_size=BATCH_SIZE,
    )
    pa_raw_results.extend(batch_results)

# Assemble prediction rows per BASELINE_SPEC.md schema
pa_prediction_rows = []
for rec, res in zip(records, pa_raw_results):
    pa_prediction_rows.append({
        'id':         rec['id'],
        'label':      rec['label'],
        'channel':    rec.get('channel'),
        'score':      res['score'],
        'pred':       res['pred'],
        'latency_ms': res['latency_ms'],
    })

write_predictions_atomic(PRED_DEBERTA, pa_prediction_rows)

# Quick score distribution
scores_pa = [r['score'] for r in pa_prediction_rows]
preds_pa  = [r['pred']  for r in pa_prediction_rows]
lats_pa   = [r['latency_ms'] for r in pa_prediction_rows]
print(f"\nProtectAI DeBERTa v2 summary:")
print(f"  n={len(scores_pa)}")
print(f"  score: min={min(scores_pa):.4f} mean={sum(scores_pa)/len(scores_pa):.4f} max={max(scores_pa):.4f}")
print(f"  pred=1 (injection): {sum(preds_pa)} / {len(preds_pa)}")
print(f"  mean latency_ms: {sum(lats_pa)/len(lats_pa):.2f}")
print(f"  Sample rows:")
for row in pa_prediction_rows[:3]:
    print(f"    {row}")

Scoring 25747 examples with protectai/deberta-v3-base-prompt-injection-v2 ...
  BATCH_SIZE=16, DEVICE=cuda


protectai-deberta:   0%|          | 0/1610 [00:00<?, ?it/s]

Written 25747 predictions → /content/drive/MyDrive/Thesis/results/baselines/predictions/protectai_deberta_v2.jsonl

ProtectAI DeBERTa v2 summary:
  n=25747
  score: min=0.0000 mean=0.1660 max=1.0000
  pred=1 (injection): 4268 / 25747
  mean latency_ms: 2.02
  Sample rows:
    {'id': 'conv-000000', 'label': 0, 'channel': None, 'score': 1.4e-05, 'pred': 0, 'latency_ms': 39.642}
    {'id': 'conv-000001', 'label': 0, 'channel': None, 'score': 7e-06, 'pred': 0, 'latency_ms': 39.642}
    {'id': 'conv-000002', 'label': 0, 'channel': None, 'score': 1.6e-05, 'pred': 0, 'latency_ms': 39.642}


## Primary: PromptGuard 2 (meta-llama/Llama-Prompt-Guard-2-86M, gated)

**Proposal reference:** Section 2.1 — _"Meta's PromptGuard 2 is one such classifier: an 86-million-parameter  
mDeBERTa-v3 model that... achieves 97.5% detection rate at 1% false positive rate on jailbreak detection."_

This model requires:
1. Accepting the license at https://huggingface.co/meta-llama/Llama-Prompt-Guard-2-86M
2. Setting `HF_TOKEN` environment variable (or running `huggingface-cli login`).

If authentication fails, we catch the error and skip to a clear printed warning.
The fallback predictions (protectai_deberta_v2) are always produced regardless.

**Label mapping for PromptGuard 2:**
- `BENIGN` (label 0) → pred=0, does not contribute to score.
- `INJECTION` (label 1) → injection from indirect data.
- `JAILBREAK` (label 2) → direct jailbreak/manipulation attempt.

Both INJECTION and JAILBREAK are treated as positive (injection) class since our eval schema  
uses binary labels covering both attack types. `score = 1 - P(BENIGN)`.

In [ ]:
PG2_MODEL_ID = 'meta-llama/Llama-Prompt-Guard-2-86M'
PG2_AVAILABLE = False  # Will be set True if load succeeds
pg2_tokenizer = None
pg2_model = None
PG2_INJECTION_INDICES = []

# Try to get HF token from env
hf_token = "#updateme"

# Attempt gated model load
try:
    from huggingface_hub import login as hf_login

    if hf_token:
        hf_login(token=hf_token, add_to_git_credential=False)
        print(f"HF_TOKEN found — attempting {PG2_MODEL_ID} ...")
    else:
        print(f"HF_TOKEN not set — attempting cached load of {PG2_MODEL_ID} ...")
        print("(If not cached, this will fail gracefully.)")

    t0 = time.perf_counter()
    pg2_tokenizer = AutoTokenizer.from_pretrained(PG2_MODEL_ID, token=hf_token)
    pg2_model = AutoModelForSequenceClassification.from_pretrained(
        PG2_MODEL_ID, token=hf_token
    )
    pg2_model = pg2_model.to(DEVICE)
    t1 = time.perf_counter()
    print(f"Loaded {PG2_MODEL_ID} in {t1-t0:.1f}s")
    print(f"Labels: {pg2_model.config.id2label}")

    pg2_id2label = pg2_model.config.id2label
    # PromptGuard 2 is a BINARY classifier: label 0 = benign, label 1 = malicious.
    # Its config.json ships WITHOUT id2label, so transformers falls back to the
    # generic {0: 'LABEL_0', 1: 'LABEL_1'} — match those names as benign/malicious.
    PG2_BENIGN_INDICES = [
        idx for idx, lbl in pg2_id2label.items()
        if lbl.upper() in ('BENIGN', 'SAFE', 'LABEL_0')
    ]
    PG2_INJECTION_INDICES = [
        idx for idx in pg2_id2label
        if idx not in PG2_BENIGN_INDICES
    ]
    # Guard against silent mis-mapping (empty benign set → every class counted
    # as injection → score ≡ 1.0 on all inputs).
    if not PG2_BENIGN_INDICES or not PG2_INJECTION_INDICES:
        raise ValueError(
            f"Label mapping failed for {PG2_MODEL_ID}: id2label={pg2_id2label} "
            f"→ benign={PG2_BENIGN_INDICES}, injection={PG2_INJECTION_INDICES}. "
            "Update the benign label-name set in this cell."
        )
    print(f"Benign indices: {PG2_BENIGN_INDICES} → {[pg2_id2label[i] for i in PG2_BENIGN_INDICES]}")
    print(f"Injection indices: {PG2_INJECTION_INDICES} → {[pg2_id2label[i] for i in PG2_INJECTION_INDICES]}")
    PG2_AVAILABLE = True

except ValueError:
    raise  # Label-mapping bug — do not swallow; fix the mapping instead of skipping.
except Exception as e:
    print()
    print("=" * 70)
    print("WARNING: PromptGuard 2 (gated) could not be loaded.")
    print(f"  Error: {type(e).__name__}: {e}")
    print()
    print("  To enable the gated model:")
    print(f"  1. Accept the license at https://huggingface.co/meta-llama/Llama-Prompt-Guard-2-86M")
    print("  2. Set environment variable: export HF_TOKEN=<your_token>")
    print("  3. Re-run this notebook.")
    print()
    print("  Continuing with protectai_deberta_v2 fallback only.")
    print("=" * 70)
    print()

HF_TOKEN found — attempting meta-llama/Llama-Prompt-Guard-2-86M ...


config.json:   0%|          | 0.00/871 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loaded meta-llama/Llama-Prompt-Guard-2-86M in 7.5s
Labels: {0: 'LABEL_0', 1: 'LABEL_1'}
Benign indices: [0] → ['LABEL_0']
Injection indices: [1] → ['LABEL_1']


In [13]:
pg2_prediction_rows = []

if PG2_AVAILABLE:
    print(f"Scoring {len(records)} examples with {PG2_MODEL_ID} ...")
    print(f"  BATCH_SIZE={BATCH_SIZE}, DEVICE={DEVICE}")

    pg2_raw_results = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='promptguard2-86m'):
        batch = texts[i : i + BATCH_SIZE]
        batch_results = batch_score(
            batch,
            pg2_tokenizer,
            pg2_model,
            PG2_INJECTION_INDICES,
            DEVICE,
            batch_size=BATCH_SIZE,
        )
        pg2_raw_results.extend(batch_results)

    for rec, res in zip(records, pg2_raw_results):
        pg2_prediction_rows.append({
            'id':         rec['id'],
            'label':      rec['label'],
            'channel':    rec.get('channel'),
            'score':      res['score'],
            'pred':       res['pred'],
            'latency_ms': res['latency_ms'],
        })

    write_predictions_atomic(PRED_PG2, pg2_prediction_rows)

    scores_pg2 = [r['score'] for r in pg2_prediction_rows]
    preds_pg2  = [r['pred']  for r in pg2_prediction_rows]
    lats_pg2   = [r['latency_ms'] for r in pg2_prediction_rows]
    print(f"\nPromptGuard 2 summary:")
    print(f"  n={len(scores_pg2)}")
    print(f"  score: min={min(scores_pg2):.4f} mean={sum(scores_pg2)/len(scores_pg2):.4f} max={max(scores_pg2):.4f}")
    print(f"  pred=1 (injection): {sum(preds_pg2)} / {len(preds_pg2)}")
    print(f"  mean latency_ms: {sum(lats_pg2)/len(lats_pg2):.2f}")
    print(f"  Sample rows:")
    for row in pg2_prediction_rows[:3]:
        print(f"    {row}")
else:
    print("PromptGuard 2 skipped (auth failure). predictions/promptguard2_86m.jsonl not written.")

Scoring 25747 examples with meta-llama/Llama-Prompt-Guard-2-86M ...
  BATCH_SIZE=16, DEVICE=cuda


promptguard2-86m:   0%|          | 0/1610 [00:00<?, ?it/s]

Written 25747 predictions → /content/drive/MyDrive/Thesis/results/baselines/predictions/promptguard2_86m.jsonl

PromptGuard 2 summary:
  n=25747
  score: min=0.0003 mean=0.1793 max=0.9996
  pred=1 (injection): 4506 / 25747
  mean latency_ms: 2.96
  Sample rows:
    {'id': 'conv-000000', 'label': 0, 'channel': None, 'score': 0.004165, 'pred': 0, 'latency_ms': 0.63}
    {'id': 'conv-000001', 'label': 0, 'channel': None, 'score': 0.054781, 'pred': 0, 'latency_ms': 0.63}
    {'id': 'conv-000002', 'label': 0, 'channel': None, 'score': 0.000527, 'pred': 0, 'latency_ms': 0.63}


## Metrics evaluation

Calls `src/evaluation/metrics.py::evaluate_detector()` if the module is implemented.  
If the module is a stub (not yet implemented by the metrics teammate), prints raw score distributions instead.  

**Proposal reference:** Section 3 (Evaluation Protocol — metrics include DR@FPR, binary F1, ECE, per-channel breakdown).

In [19]:
import importlib
metrics_file = CASCADE_ROOT / 'src' / 'evaluation' / 'metrics.py'
print("metrics.py exists:", metrics_file.exists())
importlib.invalidate_caches()

metrics.py exists: True


In [20]:
import sys

# Add src/ to path so we can import the metrics module
src_path = str(CASCADE_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

metrics_available = False
try:
    from evaluation.metrics import evaluate_detector
    # Check it's actually implemented (not a stub)
    import inspect
    src = inspect.getsource(evaluate_detector)
    if len(src.strip().splitlines()) < 5:
        raise ImportError("evaluate_detector appears to be a stub")
    metrics_available = True
    print("metrics.py: evaluate_detector() found and implemented.")
except (ImportError, AttributeError, OSError) as e:
    print(f"metrics.py not yet implemented (will print raw distributions): {e}")


def print_raw_distributions(name: str, rows: list):
    """Fallback: print score distributions when metrics.py is a stub."""
    if not rows:
        print(f"  {name}: no predictions available.")
        return
    scores = [r['score'] for r in rows if r['score'] is not None]
    preds  = [r['pred']  for r in rows]
    labels = [r['label'] for r in rows]
    n = len(rows)
    n_inj = sum(labels)
    n_ben = n - n_inj
    tp = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 1)
    fp = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 0)
    tn = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 0)
    fn = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 1)
    tpr = tp / n_inj  if n_inj > 0 else float('nan')
    fpr = fp / n_ben  if n_ben > 0 else float('nan')
    prec = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    f1   = 2*prec*tpr / (prec + tpr) if (prec + tpr) > 0 else float('nan')
    print(f"  {name}:")
    print(f"    n={n}  n_injection={n_inj}  n_benign={n_ben}")
    if scores:
        print(f"    score: min={min(scores):.4f}  mean={sum(scores)/len(scores):.4f}  max={max(scores):.4f}")
    print(f"    TP={tp}  FP={fp}  TN={tn}  FN={fn}")
    print(f"    TPR (recall) = {tpr:.4f}  FPR = {fpr:.4f}")
    print(f"    Precision = {prec:.4f}  F1 = {f1:.4f}")
    lats = [r['latency_ms'] for r in rows]
    print(f"    mean latency_ms = {sum(lats)/len(lats):.2f}")


print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

detectors_to_eval = []
if pa_prediction_rows:
    detectors_to_eval.append(('protectai_deberta_v2', PRED_DEBERTA, pa_prediction_rows))
if pg2_prediction_rows:
    detectors_to_eval.append(('promptguard2_86m', PRED_PG2, pg2_prediction_rows))

metrics_results = {}

for det_name, pred_path, pred_rows in detectors_to_eval:
    print(f"\n--- {det_name} ---")
    if metrics_available:
        try:
            out_path = METRICS_DIR / f'{det_name}.json'
            m = evaluate_detector(str(pred_path), str(out_path))
            metrics_results[det_name] = m
            print(json.dumps(m, indent=2))
        except Exception as e:
            print(f"  evaluate_detector() raised {type(e).__name__}: {e}")
            print("  Falling back to raw distributions:")
            print_raw_distributions(det_name, pred_rows)
    else:
        print_raw_distributions(det_name, pred_rows)

metrics.py: evaluate_detector() found and implemented.

EVALUATION RESULTS

--- protectai_deberta_v2 ---
{
  "detector": "protectai_deberta_v2",
  "n": 25747,
  "n_benign": 18634,
  "n_injection": 7113,
  "dr_at_fpr": {
    "0.001": {
      "dr": 0.0,
      "threshold": 1.0,
      "achieved_fpr": 0.0,
      "resolvable": true,
      "n_benign": 18634,
      "n_injection": 7113
    },
    "0.005": {
      "dr": 0.17643750878672854,
      "threshold": 0.999996,
      "achieved_fpr": 0.004829880862938714,
      "resolvable": true,
      "n_benign": 18634,
      "n_injection": 7113
    },
    "0.01": {
      "dr": 0.29312526360185576,
      "threshold": 0.999559,
      "achieved_fpr": 0.009981753783406676,
      "resolvable": true,
      "n_benign": 18634,
      "n_injection": 7113
    }
  },
  "binary": {
    "f1": 0.6194534750900624,
    "precision": 0.8259137769447048,
    "recall": 0.49557148882328134,
    "tpr": 0.49557148882328134,
    "fpr": 0.03987334979070516,
    "accuracy": 0.83

## Summary

This notebook produced:

| File | Status |
|------|--------|
| `results/baselines/predictions/protectai_deberta_v2.jsonl` | Always written (ungated) |
| `results/baselines/predictions/promptguard2_86m.jsonl` | Written if HF_TOKEN set + license accepted |
| `results/baselines/metrics/protectai_deberta_v2.json` | Written if metrics.py implemented |
| `results/baselines/metrics/promptguard2_86m.json` | Written if both gated model + metrics available |

**For the gated model (PromptGuard 2), the user must:**
1. Go to https://huggingface.co/meta-llama/Llama-Prompt-Guard-2-86M and accept the license.
2. Create an HF access token at https://huggingface.co/settings/tokens.
3. Set `export HF_TOKEN=hf_...` in the terminal before launching this notebook.
4. Re-run from the top.

In [13]:
# Final status cell
print("=" * 60)
print("NOTEBOOK COMPLETE")
print("=" * 60)
print(f"  RUN_MODE          : {RUN_MODE}")
print(f"  Records scored    : {len(records)}")
print(f"  protectai_deberta_v2.jsonl : {'✓' if PRED_DEBERTA.exists() else '✗'}")
print(f"  promptguard2_86m.jsonl     : {'✓' if PRED_PG2.exists() else '✗ (gated model not loaded)'}")

if metrics_results:
    print("\nMetrics:")
    for det, m in metrics_results.items():
        print(f"  {det}: {m}")

NOTEBOOK COMPLETE
  RUN_MODE          : full
  Records scored    : 25747
  protectai_deberta_v2.jsonl : ✓
  promptguard2_86m.jsonl     : ✓
